<a href="https://colab.research.google.com/github/omarpb03/Data-Science-ML-Fundamentals-Python/blob/main/Cross_validation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## The validation set approach


Suppose that we would like to estimate the test error associated with fitting a particular statistical learning method on a set of observations.
So we randomly divide the available set of observations into two parts, a training set and a validation set or hold-out set.
The model is fit on the training set, and the fitted model is used to predict the responses for the observations in the validation set.


In this lab, we explore the resampling techniques covered in this chapter.

In [1]:
import numpy as np
import statsmodels.api as sm
from ISLP import load_data
from ISLP.models import (ModelSpec as MS,
                         summarize,
                         poly)
from sklearn.model_selection import train_test_split

In [2]:
from functools import partial
from sklearn.model_selection import \
    (cross_validate,
     KFold,
     ShuffleSplit)
from sklearn.base import clone
from ISLP.models import sklearn_sm

In [3]:
#the validation set approach

Auto = load_data('Auto')
Auto_train, Auto_valid = train_test_split(Auto, test_size=196, random_state=0)


Now we can fit a linear regression using only the observations corresponding to the training set Auto_train

In [4]:
hp_mm=MS(['horsepower'])
X_train=hp_mm.fit_transform(Auto_train)
y_train=Auto_train['mpg']
model=sm.OLS(y_train, X_train)
results=model.fit()

In [5]:
#we now use the predict() method of results evaluated on the model matrix for this model created using
#the validation data set.
X_valid=hp_mm.fit_transform(Auto_valid)
y_valid=Auto_valid['mpg']
valid_pred=results.predict(X_valid)
np.mean((y_valid - valid_pred)**2)#we calculate the validation MSE of our model

np.float64(23.61661706966988)

Hence, our estimate for the validation MSE of the linear regresssion fit is 23.63

We can also estimate the validation error for higher-degree polynomial regressions.


In [6]:
def evalMSE(terms, response, train, test):
  mm=MS(terms)
  X_train=mm.fit_transform(train)
  y_train=train[response]
  X_test=mm.transform(test)
  y_test=test[response]
  model=sm.OLS(y_train, X_train)#OLS=Ordinary Least Squares
  results=model.fit()
  test_pred=results.predict(X_test)
  return np.mean((y_test - test_pred)**2)

Let's use this function to estimate the validation MSE using linear, quadratic and cubic fits. We use the enumerate() function here, which gives both the values and indices of objects as one iterates ver a for loop.

In [7]:
MSE=np.zeros(3)
for idx, degree in enumerate(range(1,4)):
  MSE[idx]=evalMSE([poly('horsepower', degree)], 'mpg', Auto_train, Auto_valid)
MSE

array([23.61661707, 18.76303135, 18.79694163])